In [12]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import scipy as sp
import pandas as pd
import networkx as nx
import random
import matplotlib.pyplot as plt
# import warnings filter
from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=FutureWarning)
from IPython.core.display import HTML
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="umap")

---

In [14]:

from amr.pubmed.queries import (
    build_antibiotic_pathogen_query,
    expand_synonyms,
    prepare_antibiotic_pubmed_query,
)
from amr.pubmed.search import (
    search_pmids,
    retrieve_pmids,
    fetch_pub_metadata,
    parse_pubdate,
)
from amr.pubmed.cache import (
    merge_with_existing,
    update_metadata_cache,
    get_metadata,
)
from amr.pipelines.collection import (
    fetch_pmids_intervals,
    fetch_pmids_by_date_and_keywords,
)
from amr.features.venues import add_venue_vectors
from amr.visualization import plot_venue_vector


In [15]:
antibiotics_df = pd.read_excel("antibiotic_list.xlsx")
antibiotics_list = antibiotics_df['Antibiotic'].values
print(f'#antibiotics: {len(antibiotics_list)}')

data_df = pd.read_parquet("data.parquet")
pathogens_list = data_df[data_df['n_causation']>=3]['CanonicalSpecies'].values
print(f'#pathogens: {len(pathogens_list)}')

bacteria_synonym_df = pd.read_csv("bacteria_synonym_lookup.csv")
print(f'#synonyms: {len(bacteria_synonym_df)}')

#antibiotics: 245
#pathogens: 773
#synonyms: 24932


In [16]:
data_df

,Genus,CanonicalSpecies,n_causation,n_articles,n_sampled,pathogenic_index,embedding,anonymised_embedding,onehot_embedding,domain,kingdom,phylum,class,order,family
0,Abiotrophia,Abiotrophia defectiva,6,6,6,0.160939,"[0.018194107338786125, -0.004734894765230517, ...","[0.014040807844139636, -0.028420374651129048, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.166...",Bacteria,Bacillati,Bacillota,Bacilli,Lactobacillales,Aerococcaceae
1,Acetobacter,Acetobacter indonesiensis,1,3,3,0.000000,"[-0.021512236446142197, -0.03992129862308502, ...","[-0.0028289835900068283, -0.04833690822124481,...","[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",Bacteria,Pseudomonadati,Pseudomonadota,Alphaproteobacteria,Rhodospirillales,Acetobacteraceae
2,Acetobacter,Acetobacter tropicalis,1,2,2,0.000000,"[-0.0070418259128928185, -0.005829853471368551...","[0.005147432442754507, -0.013049994595348835, ...","[0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",Bacteria,Pseudomonadati,Pseudomonadota,Alphaproteobacteria,Rhodospirillales,Acetobacteraceae
3,Achromobacter,Achromobacter animicus,1,2,2,0.000000,"[0.011074555106461048, 0.013332720845937729, 0...","[0.012643104419112206, -0.01178667601197958, 0...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ...",Bacteria,Pseudomonadati,Pseudomonadota,Betaproteobacteria,Burkholderiales,Alcaligenaceae
4,Achromobacter,Achromobacter denitrificans,75,36,36,0.387804,"[0.014773702074987037, -0.004635852294159122, ...","[0.011467029051855206, -0.012983008560258896, ...","[0.04, 0.013333333333333334, 0.066666666666666...",Bacteria,Pseudomonadati,Pseudomonadota,Betaproteobacteria,Burkholderiales,Alcaligenaceae
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1274,Yersinia,Yersinia enterocolitica,85,3419,200,0.654031,"[-0.009990866917788105, 0.009774883079979405, ...","[0.0024005417033656747, -0.009907858140479005,...","[0.0, 0.0, 0.0, 0.03529411764705882, 0.0, 0.0,...",Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Yersiniaceae
1275,Yersinia,Yersinia frederiksenii,1,21,21,0.000000,"[-0.00901515781879425, 0.02502085641026497, 0....","[0.01804814115166664, 0.005011445842683315, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Yersiniaceae
1276,Yersinia,Yersinia pestis,124,2184,200,0.647694,"[-0.006688895473886195, 0.014983039929487154, ...","[0.011833864605177364, -0.0048193380043226024,...","[0.0, 0.0, 0.0, 0.016129032258064516, 0.0, 0.0...",Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Yersiniaceae
1277,Yersinia,Yersinia pseudotuberculosis,88,1234,200,0.565611,"[-0.01641193700528344, 0.008593090970448842, 0...","[0.004612311640316461, -0.005791128928070644, ...","[0.011363636363636364, 0.0, 0.0113636363636363...",Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Yersiniaceae


In [17]:
amr_df = pd.read_parquet("pmids_results.parquet")
amr_df

,pathogen,antibiotic,start_date,end_date,pmids,count,venue_histogram
0,Abiotrophia defectiva,Spectinomycin,1975/01/01,1979/12/31,[],0,{}
1,Abiotrophia defectiva,Spectinomycin,1980/01/01,1984/12/31,[],0,{}
2,Abiotrophia defectiva,Spectinomycin,1985/01/01,1989/12/31,[],0,{}
3,Abiotrophia defectiva,Spectinomycin,1990/01/01,1994/12/31,[],0,{}
4,Abiotrophia defectiva,Spectinomycin,1995/01/01,1999/12/31,[],0,{}
...,...,...,...,...,...,...,...
1893845,Yokenella regensburgei,Trimethoprim,2000/01/01,2004/12/31,[],0,{}
1893846,Yokenella regensburgei,Trimethoprim,2005/01/01,2009/12/31,[],0,{}
1893847,Yokenella regensburgei,Trimethoprim,2010/01/01,2014/12/31,[],0,{}
1893848,Yokenella regensburgei,Trimethoprim,2015/01/01,2019/12/31,[],0,{}


In [18]:
df = pd.read_parquet("pmids_results_filtered.parquet")
df

,pmid,decision,rationale,pathogen,antibiotic,title,abstract,start_date,end_date
0,27544979,no,The abstract discusses Achromobacter xylosoxid...,Achromobacter denitrificans,Spectinomycin,Properties of Achromobacter xylosoxidans highl...,We herein discovered a highly resistant clinic...,2015/01/01,2019/12/31
1,15004037,no,The abstract mentions resistance to spectinomy...,Acinetobacter baumannii,Spectinomycin,Molecular epidemiology of sequential outbreaks...,The molecular epidemiology of multidrug-resist...,2000/01/01,2004/12/31
2,19361944,no,The abstract does not explicitly state resista...,Acinetobacter baumannii,Spectinomycin,Genetic basis of resistance to aminoglycosides...,A total of 75 Acinetobacter isolates resistant...,2005/01/01,2009/12/31
3,19028622,no,The abstract does not mention Acinetobacter ba...,Acinetobacter baumannii,Spectinomycin,Nosocomial spread of armA-mediated high-level ...,"1,310 Enterobacteriaceae, 242 Pseudomonas aeru...",2005/01/01,2009/12/31
4,16000417,yes,"The abstract mentions gene aadA1, which confer...",Acinetobacter baumannii,Spectinomycin,Detection and typing of integrons in epidemic ...,Integrons were sought in Acinetobacter isolate...,2005/01/01,2009/12/31
...,...,...,...,...,...,...,...,...,...
82775,30355751,uncertain,no abstract,Streptococcus agalactiae,Erythromycin,"Erratum for Piccinelli et al., "" In Vitro Act...",,2015/01/01,2019/12/31
82776,28060047,uncertain,LLM failure: OpenAI completion failed: Error c...,Streptococcus agalactiae,Erythromycin,Genotype Characterization of Group B Streptoco...,Group B streptococcus (GBS) is one of the lead...,2015/01/01,2019/12/31
82777,26732695,uncertain,LLM failure: OpenAI completion failed: Error c...,Streptococcus agalactiae,Erythromycin,Comparative erythromycin and tylosin susceptib...,"Tylosin, a 16-membered macrolide, is - besides...",2015/01/01,2019/12/31
82778,26415872,uncertain,LLM failure: OpenAI completion failed: Error c...,Streptococcus agalactiae,Erythromycin,Molecular epidemiology and distribution of ser...,"This study describes, for the first time, the ...",2015/01/01,2019/12/31


In [19]:
n_samples = 200
with pd.ExcelWriter("pmids_results_filtered_samples.xlsx") as xw:
    df[df['decision'].str.lower() == 'yes'].sample(n=min(n_samples, (df['decision'].str.lower() == 'yes').sum()), random_state=42).to_excel(xw, sheet_name='yes', index=False)
    df[df['decision'].str.lower() == 'no'].sample(n=min(n_samples, (df['decision'].str.lower() == 'no').sum()), random_state=42).to_excel(xw, sheet_name='no', index=False)

---

In [ ]:
from datetime import datetime
import os
import pandas as pd
from amr.pipelines.collection import count_pmids_per_year_for_pathogens_resumable

# 1) Config
EMAIL = os.getenv("NCBI_EMAIL", "your_email@example.com")  # set your contact email
START_YEAR = 1950
END_YEAR = datetime.now().year

# 2) Load synonym table if not in memory
try:
    bacteria_synonym_df  # noqa: F821
except NameError:
    bacteria_synonym_df = pd.read_csv("bacteria_synonym_lookup.csv")

# 3) Load pathogen list if not in memory
try:
    pathogens_list  # noqa: F821
except NameError:
    # Fallback: take all canonical species in data.parquet
    data_df = pd.read_parquet("data.parquet")
    pathogens_list = (
        data_df["CanonicalSpecies"]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

print(f"#pathogens to query: {len(pathogens_list)}  | years: {START_YEAR}-{END_YEAR}")

# 4) Query PubMed for per-year TIAB counts (synonym-aware) with resume/saving
counts_df = count_pmids_per_year_for_pathogens_resumable(
    pathogens_list,
    bacteria_synonym_df,
    start_year=START_YEAR,
    end_year=END_YEAR,
    email=EMAIL,
    output_path="pathogen_tiab_counts.parquet",
    save_every=200,
    progress_every=50,
)

# 5) Data is saved incrementally to 'pathogen_tiab_counts.parquet'
print(f"Current counts size: {counts_df.shape}")
display(counts_df.head(10))


#pathogens to query: 773  | years: 1950-2025
Progress: 50/58494 pairs processed… [ETA 13h 37 min]
Progress: 100/58494 pairs processed… [ETA 13h 31 min]
Progress: 150/58494 pairs processed… [ETA 13h 24 min]
Progress: 200/58494 pairs processed… [ETA 13h 9 min]


---

In [ ]:
# Antibiotic TIAB counts per year (resumable)
from amr.pipelines.collection import count_pmids_per_year_for_antibiotics_resumable

# Use antibiotics_list if present; else load from Excel
try:
    antibiotics_list  # noqa: F821
except NameError:
    import pandas as pd
    antibiotics_df = pd.read_excel("antibiotic_list.xlsx")
    antibiotics_list = antibiotics_df['Antibiotic'].dropna().astype(str).drop_duplicates().tolist()

print(f"#antibiotics to query: {len(antibiotics_list)} | years: {START_YEAR}-{END_YEAR}")

ab_counts_df = count_pmids_per_year_for_antibiotics_resumable(
    antibiotics_list,
    start_year=START_YEAR,
    end_year=END_YEAR,
    email=EMAIL,
    output_path="antibiotic_tiab_counts.parquet",
    save_every=200,
    progress_every=50,
)

print(f"Current antibiotic counts size: {ab_counts_df.shape}")
display(ab_counts_df.head(10))


---